# ContextQA - Phase 4: Fine-Tuning DistilBERT on SQuAD v1.1

In this notebook, we take the raw, pre-trained foundation model **`distilbert-base-uncased`** (which was only trained on Masked Language Modeling) and fine-tune it on the **SQuAD v1.1** dataset to create our custom Extractive QA engine.

### Hardware Optimization (NVIDIA RTX 3060 - 6GB VRAM):
- **Precision**: `fp16=True` (Halves VRAM consumption, utilizes Tensor Cores)
- **Batching**: `per_device_train_batch_size = 8` with `gradient_accumulation_steps = 2` (Effective batch size = 16)
- **Sequence Length**: `max_seq_length = 384` with `doc_stride = 128`
- **Optimizer**: `AdamW` with $\text{lr} = 3 \times 10^{-5}$, weight decay $0.01$, and $10\%$ linear warmup

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, TrainingArguments, Trainer, default_data_collator
from datasets import load_dataset
from src.preprocessing import prepare_train_features, prepare_validation_features
from src.utils import set_seed, get_device

set_seed(42)
device = get_device()
print(f"Active compute device: {device}")

## 1. Load Foundation Model & Tokenizer

In [ ]:
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)
print(f"Loaded {model_checkpoint} with {sum(p.numel() for p in model.parameters()):,} parameters.")

## 2. Load and Preprocess SQuAD Dataset

In [ ]:
raw_datasets = load_dataset("rajpurkar/squad")

train_dataset = raw_datasets["train"].map(
    lambda ex: prepare_train_features(ex, tokenizer, max_seq_length=384, doc_stride=128),
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
    desc="Tokenizing train features"
)
print(f"Prepared {len(train_dataset):,} training chunks from {len(raw_datasets['train']):,} original examples.")

## 3. Configure Trainer & Execute Fine-Tuning

In [ ]:
training_args = TrainingArguments(
    output_dir="models/distilbert_finetuned",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    save_strategy="epoch",
    save_total_limit=1,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=default_data_collator,
    tokenizer=tokenizer,
)

# Start training
# train_result = trainer.train()

## 4. Save Final Fine-Tuned Weights & Tokenizer

In [ ]:
# trainer.save_model("models/distilbert_finetuned")
# tokenizer.save_pretrained("models/distilbert_finetuned")
print("Model checkpoints saved to models/distilbert_finetuned/")